# Unsloth GRPO Training for Bitcoin Enhanced Prediction

This notebook implements Group Relative Policy Optimization (GRPO) using Unsloth for comprehensive Bitcoin prediction.

**Dataset**: `bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news`

**Training Method**: Unsloth GRPO
- Built-in preference learning optimization
- Efficient memory usage with Unsloth
- Streamlined training pipeline

## Install Libraries

In [ ]:
# !pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -U xformers trl peft accelerate bitsandbytes

In [ ]:
# Ensure protobuf uses pure-Python implementation early to avoid descriptor errors
# import os as _osThe code snippet you provided is setting environment variables using the `os.environ.setdefault()` method.

# _os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
# _os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# !pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install -U xformers trl peft accelerate bitsandbytes

## Imports

In [ ]:
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModel
from peft import PeftModel
import torch, random, os
import json
import numpy as np
from datetime import datetime


try:
    from trl import GRPOTrainer, GRPOConfig

    GRPO_AVAILABLE = True
    print("✅ GRPO classes imported successfully")
except ImportError:
    try:

        from trl import PPOTrainer, PPOConfig
        from trl import SFTTrainer, SFTConfig

        GRPO_AVAILABLE = False
        print("⚠️ GRPOTrainer not found, will use SFTTrainer as fallback")
    except ImportError:

        from transformers import Trainer

        GRPO_AVAILABLE = False
        print("⚠️ Advanced TRL classes not found, using basic Trainer")

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## Configuration

In [ ]:
BASE_MODEL_NAME = "./Qwen3-8B"
FALLBACK_MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
ADAPTER_PATH = "./my-awesome-model_final_bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news-v2"
CHECKPOINT = "checkpoint-400"
MAX_SEQ_LENGTH = 2048
DTYPE = torch.bfloat16
LOAD_IN_4BIT = True


LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


OUTPUT_DIR = "./qwen_bitcoin_enhanced_grpo_unsloth_pretrained_from_sft"
LEARNING_RATE = 3e-7
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 512
BETA = 0.1


SANITY_RUN = False
SANITY_MAX_STEPS = 30
SANITY_DATASET_SIZE = 256


DATASET_NAME = (
    "tahamajs/bitcoin-enhanced-prediction-dataset-with-local-comprehensive-news"
)


REWARD_MODEL_NAME = "microsoft/DialoGPT-medium"

## Load Model and Tokenizer

In [ ]:
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoTokenizer, AutoModel
from unsloth import FastLanguageModel, get_chat_template


preferred_path = Path(BASE_MODEL_NAME)
chosen_model_name = BASE_MODEL_NAME if preferred_path.exists() else FALLBACK_MODEL_NAME
if chosen_model_name != BASE_MODEL_NAME:
    print(
        f"ℹ️ Local model path not found at {BASE_MODEL_NAME}. Falling back to {FALLBACK_MODEL_NAME}"
    )

print(f"🔄 Loading base model: {chosen_model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=chosen_model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)


if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token


first_adapter_loaded = False
adapter_path = f"{ADAPTER_PATH}/{CHECKPOINT}"
if not Path(adapter_path).exists():
    print(
        f"ℹ️ First adapter checkpoint not found at {adapter_path}. Skipping adapter load."
    )
else:
    print(f"🔄 Loading pre-trained adapter: {adapter_path}")
    try:

        model = PeftModel.from_pretrained(model, adapter_path)
        first_adapter_loaded = True
        print(f"✅ Successfully loaded adapter from {adapter_path}")
    except Exception as e:
        err = str(e)
        print(f"⚠️ Could not load adapter on first try: {e}")

        import re

        try:
            expected_match = re.search(
                r"copying a param with shape torch\.Size\(\[(\d+),", err
            )
            current_match = re.search(r"current model is torch\.Size\(\[(\d+),", err)
            if expected_match and current_match:
                expected = int(expected_match.group(1))
                current = int(current_match.group(1))
                missing = expected - current
                if missing > 0:
                    print(
                        f"🔧 Detected vocab mismatch. Adding {missing} special token(s) to align."
                    )
                    extra_tokens = [f"<|extra_{i}|>" for i in range(missing)]
                    tokenizer.add_special_tokens(
                        {"additional_special_tokens": extra_tokens}
                    )
                    model.resize_token_embeddings(len(tokenizer))
                    model = PeftModel.from_pretrained(model, adapter_path)
                    first_adapter_loaded = True
                    print(
                        f"✅ Adapter loaded after aligning vocab size to {len(tokenizer)}"
                    )
                else:
                    print(
                        "ℹ️ Vocab sizes match but other mismatch detected. Proceeding without adapter."
                    )
            else:
                print(
                    "ℹ️ No clear vocab mismatch pattern found. Proceeding without adapter."
                )
        except Exception as e2:
            print(f"⚠️ Auto-alignment failed: {e2}. Proceeding without adapter.")


if first_adapter_loaded:
    print("✅ First adapter is loaded and active. Skipping merge step for 4-bit model.")


tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)


print("🔧 Preparing model for new LoRA adapter training...")
try:

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
    print("✅ New LoRA adapter initialized successfully for training")
except Exception as e:
    print(f"⚠️ Error initializing new LoRA adapter: {e}")
    print("ℹ️ Continuing with current model state...")


print(f"\n🔄 Loading reward model: {REWARD_MODEL_NAME}")
try:
    reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
    reward_model = AutoModel.from_pretrained(
        REWARD_MODEL_NAME,
        torch_dtype=DTYPE,
        device_map="auto",
    )
    reward_model.eval()
    print(f"✅ Reward model loaded successfully")
except Exception as e:
    print(f"⚠️ Could not load reward model, using rule-based rewards: {e}")
    reward_model = None
    reward_tokenizer = None

print(f"\n📊 Model Configuration:")
print(f"  Base model: {chosen_model_name}")
print(
    f"  First adapter (loaded, not merged): {adapter_path if first_adapter_loaded else 'Not loaded'}"
)
print(f"  New LoRA adapter initialized for training with rank: {LORA_R}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Load in 4bit: {LOAD_IN_4BIT}")
print(f"  Data type: {DTYPE}")
print(f"  Reward model: {REWARD_MODEL_NAME if reward_model else 'Rule-based only'}")

## Load and Prepare Dataset

In [ ]:

dataset = load_dataset(DATASET_NAME, split="train")
print(f"Dataset loaded: {DATASET_NAME}")
print(f"Total samples: {len(dataset):,}")


print("\n=== Sample Data ===")
sample = dataset[0]
for key, value in sample.items():
    print(f"{key}: {str(value)[:150]}{'...' if len(str(value)) > 150 else ''}")

## Format Dataset for GRPO

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer


MAX_LENGTH = 2048


def formatting_prompts_func(examples):
    """
    Format examples for GRPO training.
    GRPOTrainer expects a single 'prompt' per sample and generates completions internally.
    The reference answer is saved in a 'target' column for offline evaluation.
    """
    instructions = examples.get("instruction", [""] * len(examples.get("input", [])))
    inputs = examples.get("input", [])
    outputs = examples.get("output", [])

    prompts = []
    targets = []
    for instruction, user_input, output in zip(instructions, inputs, outputs):
        system_msg = instruction or "You are a helpful Bitcoin market analyst."
        user_msg = user_input or ""

        prompt = (
            f"<|im_start|>system\n{system_msg}<|im_end|>\n"
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        prompts.append(prompt)

        targets.append((output or "") + "<|im_end|>")

    return {"prompt": prompts, "target": targets}


print("📝 Formatting dataset for Unsloth GRPO (prompt-only mode)...")
formatted_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Formatting prompts",
)


reference_targets = formatted_dataset["target"]
formatted_dataset = formatted_dataset.remove_columns(["target"])

print(f"Formatted dataset samples: {len(formatted_dataset):,}")
print("Columns after formatting:", formatted_dataset.column_names)

print("\n=== Formatted Sample Prompt (Text) ===")
print(formatted_dataset[0]["prompt"][:500])
print("✅ Dataset text formatting complete.")


tokenizer = AutoTokenizer.from_pretrained("unsloth/mistral-7b-instruct-v0.2-bnb-4bit")


def tokenize_function(examples):

    tokenized_output = tokenizer(
        examples["prompt"],
        truncation=True,
        max_length=2048,
    )

    tokenized_output["labels"] = tokenized_output["input_ids"][:]
    return tokenized_output


print("\n⚡ Tokenizing the dataset...")
tokenized_dataset = formatted_dataset.map(
    tokenize_function, batched=True, remove_columns=["prompt"]
)

print(f"Tokenized dataset samples: {len(tokenized_dataset):,}")
print("✅ Final columns passed to trainer:", tokenized_dataset.column_names)

In [ ]:
class IdentityCollator:
    """Pass-through collator for raw prompt samples.
    Returns list[dict] unchanged so GRPOTrainer can tokenize internally."""

    def __call__(self, features):
        return features


raw_text_collator = IdentityCollator()


first = formatted_dataset[0]
print("🔍 Sample keys:", list(first.keys()))
print("Prompt length:", len(first["prompt"]))


MAX_PROMPT_CHARS = 4000
if any(
    len(r["prompt"]) > MAX_PROMPT_CHARS
    for r in formatted_dataset.select(range(min(50, len(formatted_dataset))))
):

    def truncate_func(examples):
        prompts = []
        for p in examples["prompt"]:
            if len(p) > MAX_PROMPT_CHARS:
                p = p[:MAX_PROMPT_CHARS] + "..."
            prompts.append(p)
        return {"prompt": prompts}

    print("✂️ Truncating overlong prompts for safety...")
    formatted_dataset = formatted_dataset.map(truncate_func, batched=True)
    print("✅ Truncation pass complete")

In [ ]:
def parse_trading_output(text):
    """
    Parse trading output JSON from text response.
    Expected format: {"action":"SELL","confidence":99,"stop_loss":10668.23,"take_profit":9377.95,"forecast_10d":[...]}
    """
    if not text:
        return None

    import json
    import re

    try:

        return json.loads(text.strip())
    except:
        pass

    try:

        json_pattern = r'\{[^{}]*"action"[^{}]*\}'
        matches = re.findall(json_pattern, text, re.IGNORECASE | re.DOTALL)

        if matches:

            for match in matches:
                try:

                    cleaned = match.strip()
                    return json.loads(cleaned)
                except:
                    continue

        result = {}

        action_match = re.search(r'"action"\s*:\s*"([^"]+)"', text, re.IGNORECASE)
        if action_match:
            result["action"] = action_match.group(1).upper()

        conf_match = re.search(
            r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', text, re.IGNORECASE
        )
        if conf_match:
            result["confidence"] = float(conf_match.group(1))

        sl_match = re.search(r'"stop_loss"\s*:\s*(\d+(?:\.\d+)?)', text, re.IGNORECASE)
        if sl_match:
            result["stop_loss"] = float(sl_match.group(1))

        tp_match = re.search(
            r'"take_profit"\s*:\s*(\d+(?:\.\d+)?)', text, re.IGNORECASE
        )
        if tp_match:
            result["take_profit"] = float(tp_match.group(1))

        forecast_match = re.search(
            r'"forecast_10d"\s*:\s*\[([^\]]+)\]', text, re.IGNORECASE
        )
        if forecast_match:
            try:
                forecast_str = forecast_match.group(1)
                forecast_values = [float(x.strip()) for x in forecast_str.split(",")]
                result["forecast_10d"] = forecast_values
            except:
                pass

        return result if result else None

    except Exception as e:
        return None


def calculate_forecast_similarity(resp_forecast, gt_forecast):
    """
    Calculate similarity between two forecast arrays.
    Uses multiple metrics: correlation, directional accuracy, and magnitude similarity.
    """
    if not resp_forecast or not gt_forecast:
        return 0.0

    import numpy as np

    try:

        resp_arr = np.array(
            [float(x) for x in resp_forecast if isinstance(x, (int, float))]
        )
        gt_arr = np.array(
            [float(x) for x in gt_forecast if isinstance(x, (int, float))]
        )

        if len(resp_arr) == 0 or len(gt_arr) == 0:
            return 0.0

        min_len = min(len(resp_arr), len(gt_arr))
        resp_arr = resp_arr[:min_len]
        gt_arr = gt_arr[:min_len]

        if min_len < 2:
            return 0.0

        similarity_score = 0.0

        try:
            corr = np.corrcoef(resp_arr, gt_arr)[0, 1]
            if not np.isnan(corr):
                similarity_score += abs(corr) * 0.4
        except:
            pass

        resp_directions = np.diff(resp_arr) > 0
        gt_directions = np.diff(gt_arr) > 0
        if len(resp_directions) > 0:
            directional_accuracy = np.mean(resp_directions == gt_directions)
            similarity_score += directional_accuracy * 0.3

        try:

            resp_norm = (resp_arr - np.mean(resp_arr)) / (np.std(resp_arr) + 1e-8)
            gt_norm = (gt_arr - np.mean(gt_arr)) / (np.std(gt_arr) + 1e-8)

            mse = np.mean((resp_norm - gt_norm) ** 2)
            magnitude_similarity = max(0, 1 - (mse / 4))
            similarity_score += magnitude_similarity * 0.3
        except:
            pass

        return min(1.0, max(0.0, similarity_score))

    except Exception as e:
        return 0.0


print("✅ Helper functions for structured output parsing defined")

In [ ]:
import json
import re


def parse_trading_output(response_text):
    """
    Parses a JSON object from a string, looking for the content between ```json and ```.
    """
    try:

        match = re.search(r"```json\s*([\s\S]+?)\s*```", response_text)
        if match:
            json_str = match.group(1)
            return json.loads(json_str)
    except (json.JSONDecodeError, TypeError):

        pass
    return None


def calculate_forecast_similarity(predicted, actual):
    """
    Calculates the similarity between two forecast arrays using Mean Absolute Percentage Error (MAPE).
    A lower MAPE results in a higher similarity score (reward).
    """
    if not predicted or not actual:
        return 0.0

    min_len = min(len(predicted), len(actual))
    if min_len == 0:
        return 0.0

    predicted = predicted[:min_len]
    actual = actual[:min_len]

    errors = []
    for p, a in zip(predicted, actual):
        if a > 0:
            errors.append(abs((p - a) / a))

    if not errors:
        return 0.0

    mean_absolute_percentage_error = sum(errors) / len(errors)

    similarity = max(0, 1 - mean_absolute_percentage_error)
    return similarity


def calculate_price_prediction_reward(response, ground_truth):
    """
    Calculates a reward based ONLY on the accuracy of numerical price predictions
    (stop_loss, take_profit, and forecast_10d) against the ground truth.

    The final reward is a value between 0.0 and 1.0.
    """
    response_json = parse_trading_output(response)
    ground_truth_json = parse_trading_output(ground_truth)

    if not response_json or not ground_truth_json:
        return 0.0

    price_rewards = []

    for price_field in ["stop_loss", "take_profit"]:
        resp_price = response_json.get(price_field)
        gt_price = ground_truth_json.get(price_field)

        if (
            isinstance(resp_price, (int, float))
            and isinstance(gt_price, (int, float))
            and gt_price > 0
        ):

            price_diff_pct = abs((resp_price - gt_price) / gt_price)

            price_similarity = max(0, 1 - price_diff_pct)
            price_rewards.append(price_similarity)

    resp_forecast = response_json.get("forecast_10d")
    gt_forecast = ground_truth_json.get("forecast_10d")

    if isinstance(resp_forecast, list) and isinstance(gt_forecast, list):
        forecast_similarity = calculate_forecast_similarity(resp_forecast, gt_forecast)
        price_rewards.append(forecast_similarity)

    if not price_rewards:
        return 0.0

    total_reward = sum(price_rewards) / len(price_rewards)

    return total_reward

In [ ]:
class CustomGRPOTrainer(GRPOTrainer):
    """
    Custom GRPO Trainer that integrates the enhanced reward function
    with structured output parsing for Bitcoin trading predictions.
    This version correctly processes batches and is structured robustly.
    """

    def __init__(self, reward_model=None, reward_tokenizer=None, **kwargs):

        kwargs["reward_funcs"] = [self._compute_reward_batch]

        super().__init__(**kwargs)

        self.reward_model = reward_model
        self.reward_tokenizer = reward_tokenizer

    def _compute_reward_batch(self, prompts=None, completions=None, **kwargs_inner):
        """
        This is our main reward logic, now defined as a class method.
        It correctly loops through a batch of completions and returns a list of rewards.
        """
        rewards = []
        try:

            for completion_text in completions:

                reward_score = calculate_comprehensive_prediction_reward(
                    response=completion_text,
                    ground_truth=None,
                    reward_model=self.reward_model,
                    reward_tokenizer=self.reward_tokenizer,
                )
                rewards.append(reward_score)

            return rewards

        except Exception as e:
            print(f"Warning: Error during batch reward computation: {e}")

            batch_size = len(completions) if completions is not None else 0
            return [0.5] * batch_size

    def log_reward_details(self, response, reward_score):
        """
        Log detailed reward breakdown for debugging and analysis.
        (This method requires no changes)
        """
        print(f"\n=== Reward Analysis ===")
        print(f"Response length: {len(response)} chars")
        print(f"Reward score: {reward_score:.4f}")

        response_json = parse_trading_output(response)

        if response_json:
            print(f"Structured output found:")
            print(f"  Action: {response_json.get('action', 'N/A')}")
            print(f"  Confidence: {response_json.get('confidence', 'N/A')}")
            print(f"  Stop Loss: {response_json.get('stop_loss', 'N/A')}")
            print(f"  Take Profit: {response_json.get('take_profit', 'N/A')}")
            forecast = response_json.get("forecast_10d", [])
            print(f"  Forecast: {forecast[:3]}... ({len(forecast)} values)")
        else:
            print("No structured output found - using text-based scoring")

        print("=" * 25)


print("✅ Corrected CustomGRPOTrainer created.")

## Setup GRPO Training

In [ ]:
max_steps = SANITY_MAX_STEPS if SANITY_RUN else -1

grpo_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=max_steps,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    bf16=True,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_steps=0 if SANITY_RUN else 100,
    save_strategy="no" if SANITY_RUN else "steps",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    seed=SEED,
    report_to="none",
)


print(f"🎯 Training Configuration:")

## Initialize GRPO Trainer

In [ ]:
print("🔧 Initializing GRPO Trainer with pre-tokenized dataset...")
generation_kwargs = {
    "max_new_tokens": 1024,
    "do_sample": True,
    "top_k": 50,
    "temperature": 0.7,
}

if GRPO_AVAILABLE:
    try:

        from transformers import DataCollatorForLanguageModeling

        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        grpo_trainer = CustomGRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            args=grpo_args,
            train_dataset=tokenized_dataset,
            reward_model=reward_model,
            reward_tokenizer=reward_tokenizer,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            beta=BETA,
            data_collator=data_collator,
        )
        print("✅ CustomGRPOTrainer initialized successfully")
        print(f"📊 Dataset columns: {formatted_dataset.column_names}")

    except Exception as e:
        print(f"⚠️ Failed to initialize GRPO trainer: {e}")
        print("🔄 Falling back to standard Trainer")

        from transformers import Trainer, DataCollatorForLanguageModeling

        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

        grpo_trainer = Trainer(
            model=model,
            tokenizer=tokenizer,
            args=grpo_args,
            train_dataset=formatted_dataset,
            data_collator=data_collator,
        )
        print("✅ Fallback Trainer initialized with tokenized data")
else:
    print("⚠️ GRPO not available, using standard Trainer")

    from transformers import Trainer, DataCollatorForLanguageModeling

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    grpo_trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=grpo_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )
    print("✅ Standard Trainer initialized with tokenized data")

print(f"🎯 Training ready with {type(grpo_trainer).__name__}")
print(f"📊 Training dataset: {len(formatted_dataset):,} samples")
print(f"🔧 Trainer configuration:")
print(
    f"  • Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
)
print(
    f"  • Total training steps: {len(formatted_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * (0 if SANITY_RUN else NUM_TRAIN_EPOCHS)}"
)
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Dataloader workers: 0 (avoiding multiprocessing issues)")
print(f"  • Sanity run: {SANITY_RUN}")

if SANITY_RUN:
    print(f"  • Sanity max steps: {SANITY_MAX_STEPS}")
    print("⚠️ SANITY_RUN is enabled - this will be a short test run")

In [ ]:
print("🔧 Initializing GRPO Trainer...")
print(
    "✅ Final check: Columns being passed to trainer:", tokenized_dataset.column_names
)

if GRPO_AVAILABLE:
    try:

        grpo_trainer = CustomGRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            args=grpo_args,
            train_dataset=tokenized_dataset,
            reward_model=reward_model,
            reward_tokenizer=reward_tokenizer,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            beta=BETA,
        )

        print("✅ CustomGRPOTrainer initialized successfully")
        print(f"📊 Training dataset format: {formatted_dataset.column_names}")

    except Exception as e:
        print(f"⚠️ Failed to initialize GRPO trainer: {e}")
        print("🔄 Falling back to standard trainer")

        from transformers import Trainer, DataCollatorWithPadding

        data_collator = DataCollatorWithPadding(
            tokenizer=tokenizer, padding=True, return_tensors="pt"
        )

        grpo_trainer = Trainer(
            model=model,
            tokenizer=tokenizer,
            args=grpo_args,
            train_dataset=tokenized_dataset,
            data_collator=data_collator,
        )
        print("✅ Fallback trainer initialized with DataCollatorWithPadding")

else:
    print("⚠️ GRPO not available, using standard training")
    from transformers import Trainer, DataCollatorWithPadding

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer, padding=True, return_tensors="pt"
    )

    grpo_trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=grpo_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )
    print("✅ Standard trainer initialized with DataCollatorWithPadding")

print(f"🎯 Training ready with {type(grpo_trainer).__name__}")
print(f"📊 Training dataset: {len(formatted_dataset):,} samples")
print(f"🔧 Trainer configuration:")
print(
    f"  • Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
)
print(
    f"  • Total training steps: {len(formatted_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * (0 if SANITY_RUN else NUM_TRAIN_EPOCHS)}"
)
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Sanity run: {SANITY_RUN}")

if SANITY_RUN:
    print(f"  • Sanity max steps: {SANITY_MAX_STEPS}")
    print("⚠️ SANITY_RUN is enabled - this will be a short test run")

## Start GRPO Training

In [ ]:
print("🚀 Starting Unsloth GRPO Training...")
print(
    f"Training {0 if SANITY_RUN else NUM_TRAIN_EPOCHS} epoch(s) on {len(formatted_dataset):,} samples"
)
print("=" * 60)


start_time = datetime.now()
print(f"Training started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")


trainer_stats = grpo_trainer.train()


end_time = datetime.now()
training_duration = end_time - start_time


final_loss = None
steps_done = None
try:
    final_loss = getattr(trainer_stats, "training_loss", None)
    steps_done = getattr(trainer_stats, "global_step", None)
except Exception:
    pass

print("\n" + "=" * 60)
print("🎉 GRPO Training Completed!")
print(f"Training finished at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total training time: {training_duration}")
print(f"Final training loss: {final_loss if final_loss is not None else 'N/A'}")
print(f"Training steps: {steps_done if steps_done is not None else 'N/A'}")

## Save Model

In [ ]:
print("💾 Saving trained model...")


model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

print(f"✅ Model saved to: {OUTPUT_DIR}/final_model")


training_method = "Unsloth GRPO" if GRPO_AVAILABLE else "Unsloth SFT/Basic"
training_summary = {
    "base_model_name": chosen_model_name,
    "adapter_path": f"{ADAPTER_PATH}/{CHECKPOINT}",
    "adapter_loaded": adapter_loaded,
    "dataset": DATASET_NAME,
    "training_method": training_method,
    "grpo_available": GRPO_AVAILABLE,
    "total_samples": len(formatted_dataset),
    "training_config": {
        "epochs": training_config.num_train_epochs,
        "max_steps": (
            training_config.max_steps if training_config.max_steps > 0 else None
        ),
        "learning_rate": training_config.learning_rate,
        "batch_size": training_config.per_device_train_batch_size,
        "gradient_accumulation_steps": training_config.gradient_accumulation_steps,
        "sanity_run": SANITY_RUN,
        "sanity_max_steps": SANITY_MAX_STEPS if SANITY_RUN else None,
        "sanity_dataset_size": SANITY_DATASET_SIZE if SANITY_RUN else None,
    },
    "training_results": {
        "final_loss": final_loss,
        "total_steps": steps_done,
        "training_duration": str(training_duration),
    },
    "timestamps": {
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
    },
    "model_path": f"{OUTPUT_DIR}/final_model",
}


if GRPO_AVAILABLE and hasattr(training_config, "beta"):
    training_summary["training_config"]["grpo_beta"] = training_config.beta
    training_summary["training_config"]["max_length"] = training_config.max_length
    training_summary["training_config"][
        "max_prompt_length"
    ] = training_config.max_prompt_length


with open(f"{OUTPUT_DIR}/training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)

print(f"Training summary saved to: {OUTPUT_DIR}/training_summary.json")